In [1]:
import os
import numpy as np
import pandas as pd
import yfinance as yf

def mensual_desde_yf(ticker, prefijo):
    # Descarga los datos históricos
    df = yf.download(ticker, start="1990-01-01", auto_adjust=False,
                     multi_level_index=False)
    
    # Rendimiento logarítmico diario 
    df["ret_d"] = np.log(df["Close"]).diff()

    # 1) CALENDARIO CONTINUO Y RELLENO (NUEVO)
    # Crea un calendario que incluye absolutamente todos los días 
    cal = pd.date_range(df.index.min(), df.index.max(), freq="D")
    
    # Ajusta los precios de cierre al nuevo calendario. Si el día no hubo bolsa,
    # copia (ffill - forward fill) el precio de cierre del día anterior conocido.
    close_cal = df["Close"].reindex(cal).ffill()

    # 2) CIERRE EXACTO AL FIN DE MES CALENDARIO
    # Resample ("ME" - Month End) toma el valor exactamente del último día del mes (28, 29, 30 o 31).
    # Como los fines de semana ya están rellenos, nunca habrá valores nulos.
    cierre = close_cal.resample("ME").last().rename("cierre")

    # 3) ESTADÍSTICAS INTRAMENSUALES (SOLO DÍAS HÁBILES)
    # Agrupa los días que SÍ hubo bolsa para calcular mínimos, máximos y volatilidad.
    g = df.groupby(df.index.to_period("M"))
    est = g.agg(
        min_mes=("Low", "min"),
        max_mes=("High", "max"),
        vol_mes=("ret_d", "std"),
        promedio=("Close", "mean"),

    )
    
    # Guarda qué día fue la última sesión de bolsa de ese mes (ej. viernes 30)
    est["fecha_ultimo_habil"] = df.index.to_series().groupby(df.index.to_period("M")).max()
    
    # Iguala el formato del índice al de 'cierre' (último día calendario) para poder unirlos
    est.index = est.index.to_timestamp(how="end").normalize()

    # Une el cierre rellenado con las estadísticas del mes en una sola tabla
    m = pd.concat([cierre, est], axis=1)
    
    # Calcula métricas mensuales finales
    m["ret_m"] = np.log(m["cierre"]).diff()
    m["rango_mes"] = m["max_mes"] / m["min_mes"] - 1

    # 4) FILTRO DEL MES EN CURSO
    # Elimina el mes actual incompleto. Al comparar contra 'today()', 
    # descarta cualquier fila cuyo fin de mes proyectado sea mayor a la fecha de hoy.
    m = m[m.index <= pd.Timestamp.today().normalize()]
    m.index.name = "fecha"

    return m.add_prefix(f"{prefijo}_")


# --- EXTRACCIÓN E INTEGRACIÓN ---

sp = mensual_desde_yf("^GSPC", "sp500")
vix = mensual_desde_yf("^VIX", "vix")[["vix_cierre"]]

# Une S&P 500 y VIX, y pasa la 'fecha' de índice a columna normal
mercado = sp.join(vix).reset_index()

# Convierte a DataFrame de Spark
# sdf = spark.createDataFrame(mercado)

os.makedirs("../data/raw", exist_ok=True)
mercado.to_csv("../data/raw/s&p500_mensual.csv", index=False)

c:\Users\julia\anaconda3\envs\pyspark_env\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
